# TP — Stacking From Scratch - SOLUTION

**Objectif :** Implémenter une stratégie de stacking *from scratch* en réalisant soi‑même la validation croisée K‑fold (out‑of‑fold predictions), construire la matrice niveau‑1, puis entraîner un méta‑classifieur.



---


## 1) Chargement du jeu de données et split final
Nous utiliserons le jeu `breast_cancer` (classification binaire) de scikit‑learn. Séparez un jeu test final (ex. 80/20) — le test final **ne doit pas** être vu pendant la construction des out‑of‑fold predictions.


In [ ]:
# Cellule exécutable: chargement et split final
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Train size:', X_train.shape[0], 'Test size:', X_test.shape[0])


## 2) Implémenter `make_folds`
Afin de forcer les étudiants à comprendre la mécanique, implémentez la fonction `make_folds(y, k, stratify=True, shuffle=True, seed=0)` **sans** utiliser `KFold` ou `StratifiedKFold` de scikit‑learn.

**Objectif de la fonction :** retourner une liste de `k` listes d'indices (chaque liste contient les indices de validation pour ce fold).


In [ ]:
# SOLUTION: implémentation make_folds
import numpy as np

def make_folds(y, k=5, stratify=True, shuffle=True, seed=0):
    """
    Entrées:
      y : array-like (n_samples,)
      k : int, nombre de folds
      stratify : bool, si True, distribue approximativement chaque classe dans les folds
      shuffle : bool
      seed : int
    Retourne:
      folds : list de k listes d'indices (validation indices)
    """
    y = np.array(y)
    n = len(y)
    idx = np.arange(n)

    if shuffle:
        np.random.seed(seed)
        np.random.shuffle(idx)

    folds = [[] for _ in range(k)]

    if stratify:
        # Pour chaque classe, on distribue ses échantillons équitablement dans les k folds
        classes = np.unique(y)
        for c in classes:
            c_idx = idx[y[idx] == c]  # indices de cette classe
            for i, val in enumerate(c_idx):
                folds[i % k].append(val)  # on balance entre les folds
    else:
        # Sans stratification, on distribue simplement les indices
        for i, val in enumerate(idx):
            folds[i % k].append(val)  # juste couper en k morceaux

    return folds

# Test de la fonction
test_folds = make_folds(y_train, k=5, stratify=True, shuffle=True, seed=0)
print(f"Nombre de folds: {len(test_folds)}")
print(f"Taille de chaque fold: {[len(f) for f in test_folds]}")
print(f"Total d'échantillons: {sum(len(f) for f in test_folds)}")


## 3) Construire les prédictions out‑of‑fold (OOF)
Pour chaque base learner, effectuez K entraînements (K‑1 folds pour l'entraînement et 1 fold pour validation) et écrivez les probabilités/étiquettes prédites pour le fold de validation dans le vecteur OOF correspondant.


In [ ]:
# SOLUTION: implémentation build_oof_predictions
import sklearn.base

def build_oof_predictions(X, y, models, k=5, stratify=True, seed=0, proba=True):
    """
    Retourne:
      oof_preds : array (n_samples, n_models) contenant, pour chaque échantillon, 
                  la prediction out-of-fold du modèle (probabilité classe 1 si proba=True)
      fitted_models_for_full_train : dict name -> model entrainés sur tout l'ensemble
    """
    n_samples = X.shape[0]
    n_models = len(models)
    
    # Initialiser la matrice OOF avec des zéros
    oof_preds = np.zeros((n_samples, n_models))
    
    # Créer les folds
    folds = make_folds(y, k=k, stratify=stratify, shuffle=True, seed=seed)
    
    # Pour chaque modèle
    for model_idx, (model_name, model) in enumerate(models):
        print(f"\nTraitement du modèle: {model_name}")
        
        # Pour chaque fold
        for fold_idx, val_indices in enumerate(folds):
            # Créer les indices d'entraînement (tous sauf le fold de validation)
            train_indices = []
            for i, fold in enumerate(folds):
                if i != fold_idx:
                    train_indices.extend(fold)
            
            train_indices = np.array(train_indices)
            val_indices = np.array(val_indices)
            
            # Extraire les données d'entraînement et de validation
            X_fold_train = X[train_indices]
            y_fold_train = y[train_indices]
            X_fold_val = X[val_indices]
            
            # Cloner le modèle pour éviter les interférences
            model_clone = sklearn.base.clone(model)
            
            # Entraîner le modèle sur le fold d'entraînement
            model_clone.fit(X_fold_train, y_fold_train)
            
            # Prédire sur le fold de validation
            if proba:
                # Probabilité de la classe positive (classe 1)
                preds = model_clone.predict_proba(X_fold_val)[:, 1]
            else:
                preds = model_clone.predict(X_fold_val)
            
            # Stocker les prédictions OOF
            oof_preds[val_indices, model_idx] = preds
            
            print(f"  Fold {fold_idx + 1}/{k} - Train: {len(train_indices)}, Val: {len(val_indices)}")
    
    # Entraîner chaque modèle sur l'ensemble complet pour les prédictions finales
    fitted_models_for_full_train = {}
    print("\nEntraînement des modèles sur l'ensemble complet...")
    for model_name, model in models:
        model_clone = sklearn.base.clone(model)
        model_clone.fit(X, y)
        fitted_models_for_full_train[model_name] = model_clone
        print(f"  {model_name} entraîné")
    
    return oof_preds, fitted_models_for_full_train


## 4) Choix des modèles
Nous proposons 3 base learners simples et un meta-model :
- LogisticRegression (base)
- DecisionTreeClassifier (base)
- KNeighborsClassifier (base)
- LogisticRegression (méta)


In [ ]:
# modèles proposés
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

base_models = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('dt', DecisionTreeClassifier(max_depth=5)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]
meta_model = LogisticRegression()
print('Base models:', [n for n,_ in base_models])


## 5) Construire la matrice niveau‑1, entraîner le méta‑modèle et évaluer

Nous allons:
1. Construire la matrice OOF (out-of-fold) pour l'entraînement du méta-modèle
2. Entraîner le méta-modèle sur cette matrice
3. Créer la matrice de test en utilisant les modèles entraînés sur tout X_train
4. Évaluer les performances


In [ ]:
# Fonction helper pour créer les prédictions de test
def train_bases_and_predict_full(X_train, y_train, X_test, models, proba=True):
    """
    Entraîne chaque modèle de base sur tout X_train et prédit sur X_test.
    Retourne une matrice (n_test_samples, n_models)
    """
    n_test = X_test.shape[0]
    n_models = len(models)
    meta_X_test = np.zeros((n_test, n_models))
    
    for model_idx, (model_name, model) in enumerate(models):
        # Cloner et entraîner sur tout X_train
        model_clone = sklearn.base.clone(model)
        model_clone.fit(X_train, y_train)
        
        # Prédire sur X_test
        if proba:
            preds = model_clone.predict_proba(X_test)[:, 1]
        else:
            preds = model_clone.predict(X_test)
        
        meta_X_test[:, model_idx] = preds
        print(f"Modèle {model_name} - prédictions sur test générées")
    
    return meta_X_test


In [ ]:
# SOLUTION: Pipeline complet de stacking
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

print("="*60)
print("ÉTAPE 1: Construction de la matrice OOF (Out-Of-Fold)")
print("="*60)
oof_X, fitted_full = build_oof_predictions(
    X_train, y_train, base_models, k=5, stratify=True, seed=1, proba=True
)

print(f"\nMatrice OOF créée: {oof_X.shape}")
print(f"Colonnes: {[name for name, _ in base_models]}")

print("\n" + "="*60)
print("ÉTAPE 2: Entraînement du méta-modèle")
print("="*60)
meta_model.fit(oof_X, y_train)
print("Méta-modèle entraîné avec succès!")

print("\n" + "="*60)
print("ÉTAPE 3: Création de la matrice de test")
print("="*60)
meta_X_test = train_bases_and_predict_full(
    X_train, y_train, X_test, base_models, proba=True
)
print(f"\nMatrice de test créée: {meta_X_test.shape}")

print("\n" + "="*60)
print("ÉTAPE 4: Évaluation des performances")
print("="*60)

# Prédictions du stacking
y_pred_meta = meta_model.predict(meta_X_test)
y_pred_meta_proba = meta_model.predict_proba(meta_X_test)[:, 1]

print("\n--- STACKING (Méta-modèle) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_meta):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_meta_proba):.4f}")

# Comparaison avec les modèles de base individuels
print("\n--- MODÈLES DE BASE (individuels) ---")
for model_name, model in base_models:
    model_clone = sklearn.base.clone(model)
    model_clone.fit(X_train, y_train)
    y_pred = model_clone.predict(X_test)
    y_pred_proba = model_clone.predict_proba(X_test)[:, 1]
    
    print(f"\n{model_name.upper()}:")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\n" + "="*60)
print("RAPPORT DE CLASSIFICATION DÉTAILLÉ (Stacking)")
print("="*60)
print(classification_report(y_test, y_pred_meta, target_names=['Malignant', 'Benign']))


## 6) Analyse des résultats

Le stacking combine les prédictions de plusieurs modèles de base pour créer un modèle plus robuste.

**Principe:**
1. Les modèles de base (LR, DT, KNN) font des prédictions out-of-fold sur les données d'entraînement
2. Ces prédictions forment une nouvelle matrice de features (niveau 1)
3. Un méta-modèle apprend à combiner ces prédictions de manière optimale
4. Pour le test, chaque modèle de base prédit, et le méta-modèle combine ces prédictions

**Avantages:**
- Combine les forces de différents algorithmes
- Généralement plus performant que les modèles individuels
- Réduit le sur-apprentissage grâce à la validation croisée


In [ ]:
# Visualisation optionnelle des prédictions OOF
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
model_names = [name for name, _ in base_models]

for idx, (ax, model_name) in enumerate(zip(axes, model_names)):
    ax.hist(oof_X[y_train == 0, idx], bins=30, alpha=0.5, label='Malignant', color='red')
    ax.hist(oof_X[y_train == 1, idx], bins=30, alpha=0.5, label='Benign', color='blue')
    ax.set_xlabel('Probabilité prédite')
    ax.set_ylabel('Fréquence')
    ax.set_title(f'Prédictions OOF - {model_name.upper()}')
    ax.legend()

plt.tight_layout()
plt.show()

print("\nCorrélation entre les prédictions des modèles de base:")
oof_df = pd.DataFrame(oof_X, columns=model_names)
print(oof_df.corr())
